# **Feature extraction** 

### Dependencies

In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np
import time
import os
import statsmodels
import seaborn as sns
import matplotlib.pyplot as plt
import glob
from scipy import stats
import pingouin as pg
#import scikit_posthocs as sp
import scipy as sp
import statsmodels.api as sm
from scipy.stats import boxcox
from statannotations.Annotator import Annotator
from scipy.stats import pearsonr
from sklearn.preprocessing import normalize
import statsmodels.formula.api as smf
from functools import reduce
from scipy.stats import shapiro
from scipy.special import logit, expit

### Load dasets 

In [ ]:
lat_df_gamma = pd.read_csv("../../datasets/eeg/whole_brain_datasets/lateralization_power/2025_whole_brain_latpwr_gamma.csv") 
frontal_power_6 = pd.read_csv("../../datasets/eeg/region_specific_datasets/frontal_power/eegip_6_frontal_multitaper_09_03_2024.csv")
lat_df_region = pd.read_csv("../../datasets/eeg/region_specific_datasets/lateralization_power/2025_region_latpwr_gamma.csv")  
con_df_gamma = pd.read_csv("../../datasets/eeg/whole_brain_datasets/connectivity/2024_whole_brain_con_gamma.csv")  
con_df_region_theta = pd.read_csv("../../datasets/eeg/region_specific_datasets/connectivity/2024_region_con_theta.csv")
con_df_region_gamma = pd.read_csv("../../datasets/eeg/region_specific_datasets/connectivity/2024_region_con_gamma.csv")
con_df_region_alpha = pd.read_csv("../../datasets/eeg/region_specific_datasets/connectivity/2024_region_con_alpha.csv")
frontal_power_6_sources= pd.read_csv("../../datasets/eeg/region_specific_datasets/frontal_power/sources_power_6mo.csv")
frontal_power_12_sources= pd.read_csv("../../datasets/eeg/region_specific_datasets/frontal_power/sources_power_12mo.csv")
frontal_power_18_sources= pd.read_csv("../../datasets/eeg/region_specific_datasets/frontal_power/sources_power_18mo.csv")
behv_df = pd.read_csv("../../datasets/behaviour/language_datasets/source/EEGIP_msel_20210510.csv")   
asd_df = pd.read_csv("../../datasets/behaviour/demographics/demographics_eegip_2024.csv")  
demo_df = pd.read_csv("../../datasets/behaviour/demographics/source/mastersheet_latest.csv") 
clutering_solutions_df = pd.read_csv("../../datasets/eeg/clustering_df.csv") 

## Clean up the demograhics dataset

In [ ]:
demo_df=demo_df.rename(columns = {'ID':'subject','Risk Group':'group_type', 'Site':'site', 'High-risk Outcome':'alternative_outcome'}) #Sutbset by important columns
demo_df=demo_df[['subject','site',"group_type", "outcome","sex","alternative_outcome"]]
demo_df['group_type'] = demo_df['group_type'].replace({'High Risk': 'ELA', 'Low Risk': 'TLA'}) # Convert High Risk to HRA and Low Risk to LRC
demo_df['subject'] = demo_df['subject'].replace({'EC': ''}, regex=True) 
demo_df["site"] = demo_df["site"].str.lower() # Convert site to lowercase
demo_df = demo_df.replace({'777': np.nan}, regex=True) # Code 777 means missing data so replace with NaN
demo_df = demo_df.replace({777: np.nan}, regex=True) # Code 777 means missing data so replace with NaN
demo_df['alternative_outcome'] = demo_df['alternative_outcome'].replace({3: 'asd',2:'no-asd', 0:'no-asd', 1:'no-asd'}, regex=True) 
demo_df['outcome'] = demo_df['outcome'].fillna(demo_df['alternative_outcome']) # Fill missing outcome with alternative outcome
demo_df = demo_df.drop(columns=['alternative_outcome']) # Drop alternative outcome
demo_df

In [ ]:
demo_df[['site', 'group_type', 'outcome','sex']].value_counts()

In [ ]:
# Sanity check for unique values
for column in demo_df.columns:
    print(column, demo_df[column].unique()) # Check unique values in each column

In [ ]:
demo_df.to_csv("../../datasets/behaviour/demographics/demographics_eegip_2026.csv", index=False) # Save the updated demographics file

### Functions

In [ ]:
# Create function that removes outliers from a dataframe and plots the histograms before and after removing outliers
def plot_and_remove_outliers(df, column, z_threshold=3, transform=False, connectivity=False, sources=False, omitnans=False):
    
    # Check if the DataFrame is empty
    if len(df) == 0:
        return "DataFrame is empty, no data to process."
    
    fig, axs = plt.subplots(1, 2, figsize=(15, 5))

    # Flag if transformation is needed
    if shapiro(df[column])[1] < 0.05:
        print("$Data is not normal$")
        transform = True

    print("Shapiro normality test before: ", shapiro(df[column]))
    
    # Shapiro-Wilk normality test before removing outliers
    print("Number of participants before:", len(df))
     # Histogram before removing outliers
    sns.histplot(df[column], kde=True, ax=axs[0])
    axs[0].set_title("Before transforming")
    if transform and sources:
        print("No transformation needed for sources")
        transform = False
    if transform and not connectivity:
        df[column] = np.log(df[column])

    if transform and connectivity:
        # Do special logit transformation for connectivity data
        df[column] = logit(df[column])
    

    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df_filtered = df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]
    print("Number of participants before:", len(df_filtered))

    # Histogram after removing outliers
    sns.histplot(df_filtered[column], kde=True, ax=axs[1])
    axs[1].set_title("After transforming and removing outliers")
    
    plt.show()

    # Check if data is normal after removing outliers
    if shapiro(df_filtered[column])[1] < 0.05:
        print("****Data still not normal****")
        transform = True
    
    return df_filtered

## EEG Measures based on literature

## Connectivity measures

In [ ]:
# Create a dictionary with all possible combinations of frontal regions with central regions and frontal regions with ocipital regions

short_range_pairs={  
 'parsopercularis-insula-lh': ('parsopercularis-lh',
'insula-lh'),
 'parstriangularis-insula-lh': ('parstriangularis-lh',
  'insula-lh'),
  'parsorbitalis-insula-lh': ('parsorbitalis-lh',
  'insula-lh'),
 'rostralmiddlefrontal-lh_precentral-lh': ('rostralmiddlefrontal-lh',
  'precentral-lh'),
 'rostralmiddlefrontal-lh_postcentral-lh': ('rostralmiddlefrontal-lh',
  'postcentral-lh'),
   'lateralorbitofrontal-lh_insula-lh': ('lateralorbitofrontal-lh',
  'insula-lh')}
long_range_pairs = {'parsopercularis-lh_cuneus-lh': ('parsopercularis-lh', 'cuneus-lh'),
 'parsopercularis-lh_lingual-lh': ('parsopercularis-lh', 'lingual-lh'),
 'parsopercularis-lh_lateraloccipital-lh': ('parsopercularis-lh',
  'lateraloccipital-lh'),
 'parstriangularis-lh_cuneus-lh': ('parstriangularis-lh', 'cuneus-lh'),
 'parstriangularis-lh_lingual-lh': ('parstriangularis-lh', 'lingual-lh'),
 'parstriangularis-lh_lateraloccipital-lh': ('parstriangularis-lh',
  'lateraloccipital-lh'),
  'parsorbitalis-lh_cuneus-lh': ('parsorbitalis-lh', 'cuneus-lh'),
 'parsorbitalis-lh_lingual-lh': ('parsorbitalis-lh', 'lingual-lh'),
 'parsorbitalis-lh_lateraloccipital-lh': ('parsorbitalis-lh',
  'lateraloccipital-lh'),
  'rostralmiddlefrontal-lh_cuneus-lh': ('rostralmiddlefrontal-lh', 'cuneus-lh'),
 'rostralmiddlefrontal-lh_lingual-lh': ('rostralmiddlefrontal-lh',
  'lingual-lh'),
 'rostralmiddlefrontal-lh_lateraloccipital-lh': ('rostralmiddlefrontal-lh',
  'lateraloccipital-lh'),
 'rostralmiddlefrontal-lh_pericalcarine-lh': ('rostralmiddlefrontal-lh',
  'pericalcarine-lh'),
 'lateralorbitofrontal-lh_cuneus-lh': ('lateralorbitofrontal-lh', 'cuneus-lh'),
 'lateralorbitofrontal-lh_lateraloccipital-lh': ('lateralorbitofrontal-lh',
  'lateraloccipital-lh')}

frontal_regions = ['parsopercularis-lh','parstriangularis-lh', 'parsorbitalis-lh','rostralmiddlefrontal-lh',
                              'lateralorbitofrontal-lh', 'rostralmiddlefrontal-lh','caudalmiddlefrontal-lh', "frontalpole-lh"]
central_regions = ['precentral-lh',"postcentral-lh",'superiortemporal-lh','insula-lh']
occipital_regions = ['cuneus-lh','lingual-lh','lateraloccipital-lh','pericalcarine-lh'] 


#### **Measure set 1** #1: Frontal gamma power (PSD)

##### Power at 6 months

In [ ]:
frontal_power_6_sources["source_psd"]= (frontal_power_6_sources["left_psd"]+ frontal_power_6_sources["right_psd"])/2
# Based on previous anlayses, the top regions that capture frontal gamma are the "Superior frontal gyrus" and the "Rostral middle frontal Gyrus"
frontal_power_6_sources= frontal_power_6_sources.loc[frontal_power_6_sources["region1"].isin(["rostralmiddlefrontal-rh","superiorfrontal-rh"])].groupby(["subject"]).mean(numeric_only=True).reset_index()
# Rename sources_psd to front_gamma_6
frontal_power_6_sources.rename(columns={"source_psd":"front_gamma_6"}, inplace=True)
frontal_power_6_sources= frontal_power_6_sources[["subject","front_gamma_6"]]
frontal_power_6_sources=plot_and_remove_outliers(frontal_power_6_sources, "front_gamma_6", z_threshold=3, transform=False, connectivity=False, sources= True)
frontal_power_6_sources

#### Power at 12 months

In [ ]:
frontal_power_12_sources["source_psd"]= (frontal_power_12_sources["left_psd"]+ frontal_power_12_sources["right_psd"])/2
# Based on previous anlayses, the top regions that capture frontal gamma are the "Superior frontal gyrus" and the "Rostral middle frontal Gyrus"
frontal_power_12_sources= frontal_power_12_sources.loc[frontal_power_12_sources["region1"].isin(["rostralmiddlefrontal-rh","superiorfrontal-rh"])].groupby(["subject"]).mean(numeric_only=True).reset_index()
# Rename sources_psd to front_gamma_12
frontal_power_12_sources.rename(columns={"source_psd":"front_gamma_12"}, inplace=True)
frontal_power_12_sources= frontal_power_12_sources[["subject","front_gamma_12"]]
frontal_power_12_sources=plot_and_remove_outliers(frontal_power_12_sources, "front_gamma_12", z_threshold=3, transform=False, connectivity=False, sources= True)
frontal_power_12_sources


#### Power at 18 months

In [ ]:
frontal_power_18_sources["source_psd"]= (frontal_power_18_sources["left_psd"]+ frontal_power_18_sources["right_psd"])/2
# Based on previous anlayses, the top regions that capture frontal gamma are the "Superior frontal gyrus" and the "Rostral middle frontal Gyrus"
frontal_power_18_sources= frontal_power_18_sources.loc[frontal_power_18_sources["region1"].isin(["rostralmiddlefrontal-rh","superiorfrontal-rh"])].groupby(["subject"]).mean(numeric_only=True).reset_index()
# Rename sources_psd to front_gamma_18
frontal_power_18_sources.rename(columns={"source_psd":"front_gamma_18"}, inplace=True)
frontal_power_18_sources= frontal_power_18_sources[["subject","front_gamma_18"]]
frontal_power_18_sources=plot_and_remove_outliers(frontal_power_18_sources, "front_gamma_18", z_threshold=3, transform=False, connectivity=False, sources= True)
frontal_power_18_sources


#### **Measure #2**: Gamma lateralization

#### Lateralization at 6mo

In [ ]:
lat_df_gamma_6= lat_df_gamma.loc[lat_df_gamma.age == 6]
lat_df_gamma_6.rename(columns={"l_index":"gamma_lat_6"}, inplace=True)
lat_df_gamma_6=lat_df_gamma_6[["subject","gamma_lat_6"]]
lat_df_gamma_6

#### Lateralization at 12mo

In [ ]:
lat_df_gamma_12= lat_df_gamma.loc[lat_df_gamma.age == 12]
lat_df_gamma_12.rename(columns={"l_index":"gamma_lat_12"}, inplace=True)
lat_df_gamma_12=lat_df_gamma_12[["subject","gamma_lat_12"]]
lat_df_gamma_12

#### Lateralization at 18mo

In [ ]:
lat_df_gamma_18= lat_df_gamma.loc[lat_df_gamma.age == 18]
lat_df_gamma_18.rename(columns={"l_index":"gamma_lat_18"}, inplace=True)
lat_df_gamma_18=lat_df_gamma_18[["subject","gamma_lat_18"]]
lat_df_gamma_18

In [ ]:
# handle outliers

lat_df_gamma_6=plot_and_remove_outliers(lat_df_gamma_6, "gamma_lat_6", z_threshold=3, transform=False)

In [ ]:
lat_df_gamma_12=plot_and_remove_outliers(lat_df_gamma_12, "gamma_lat_12", z_threshold=3, transform=False)

In [ ]:
lat_df_gamma_18=plot_and_remove_outliers(lat_df_gamma_18, "gamma_lat_18", z_threshold=3, transform=False)

In [ ]:
con_df_region_gamma

### ***Measure #3-5** Connectivity for different regions

In [ ]:
lat_rois= {   
"dmn":[ # right hemisphere
     "caudalanteriorcingulate-rh",    'parahippocampal-rh','isthmuscingulate-rh',
            'medialorbitofrontal-rh','posteriorcingulate-rh',
       'precuneus-rh', 'rostralanteriorcingulate-rh', 'lateralorbitofrontal-rh', 
       # left hemisphere
         "caudalanteriorcingulate-lh",    'parahippocampal-lh','isthmuscingulate-lh',
                'medialorbitofrontal-lh','posteriorcingulate-lh' 
       ],
"attn":[ # right hemisphere
     'inferiorparietal-rh'   , 'superiorparietal-rh', 'temporalpole-rh' ,
             'rostralmiddlefrontal-rh','caudalmiddlefrontal-rh','supramarginal-rh','insula-rh'
             
         # left hemisphere
        'inferiorparietal-lh'   , 'superiorparietal-lh', 'temporalpole-lh' ,
                'rostralmiddlefrontal-lh','caudalmiddlefrontal-lh','supramarginal-lh','insula-lh'
             ],
"visual": [ # right hemisphere
    
    'cuneus-rh','fusiform-rh','lingual-rh','lateraloccipital-rh'
     # left hemisphere
        'cuneus-lh','fusiform-lh','lingual-lh','lateraloccipital-lh'
    ] ,

"auditory": ["superiortemporal-rh",  "transversetemporal-rh"
                "superiortemporal-lh", "transversetemporal-lh"
             ],
 
 # Source: Yuan, Binke, et al. "The domain-separation language network dynamics in resting state support its flexible functional
 #  segregation and integration during language and speech processing." NeuroImage 274 (2023): 120132.
"lang": [ 
          # left hemisphere
        'parsopercularis-lh', 'parsorbitalis-lh', 'parstriangularis-lh', # IFG
                    "middletemporal-lh", "superiortemporal-lh", # Temporal
            "supramarginal-lh", "inferiorparietal-lh" # Parietal
        ],
"lang_comp": [# Bilateral MTG
        'middletemporal-lh', 'middletemporal-rh',

        # Bilateral IPL
        'inferiorparietal-lh', 'inferiorparietal-rh',

        # Bilateral Supramarginal
        'supramarginal-lh', 'supramarginal-rh',

        # Bilateral IFG
        'parsopercularis-lh', 'parsopercularis-rh',
        'parstriangularis-lh', 'parstriangularis-rh',
        'parsorbitalis-lh', 'parsorbitalis-rh'], 

"speech": [
    # Left IFG
        'parsopercularis-lh',  
        'parstriangularis-lh', 
        'parsorbitalis-lh',  

        # Left Precentral 
        'precentral-lh',  

        # Left STG     
          'superiortemporal-lh'
        ]}
for network, region in lat_rois.items():
    condition = con_df_region_gamma['source'].isin(region) & con_df_region_gamma['target'].isin(region)
    con_df_region_gamma[network] = condition
    print(network, region)  

# Initialize an empty list to collect dataframes for merging later
all_dfs = []

# Loop over each network in lat_rois
for network in lat_rois.keys():
    # For both age groups (6 and 12 months)
    for age in [6, 12, 18]:
        # Create a new column for the total connectivity (both left and right)
        con_df_age = con_df_region_gamma.loc[con_df_region_gamma.age == age]
        con_df_age = con_df_age[con_df_age[network]]  # Filter by network connectivity
        con_df_age=con_df_age.groupby(["subject"]).mean(numeric_only=True).reset_index()
        con_df_age.rename(columns={"connectivity": f"{network}_con_{age}"}, inplace=True)
        con_df_age = con_df_age[["subject", f"{network}_con_{age}"]]

        # handle outliers
        con_df_age=plot_and_remove_outliers(con_df_age, f"{network}_con_{age}", z_threshold=3, 
                                            transform=False, connectivity = True)
        all_dfs.append(con_df_age)
        
        if network=="auditory":
            continue
        # Left hemisphere connectivity for the current age group
        con_df_age_left = con_df_region_gamma.loc[con_df_region_gamma.age == age]
        con_df_age_left = con_df_age_left[con_df_age_left[network]]
        con_df_age_left = con_df_age_left[con_df_age_left.hem_con == "left"]  # Filter left hemisphere
        con_df_age_left = con_df_age_left.groupby(["subject"]).mean(numeric_only=True).reset_index()
        con_df_age_left.rename(columns={"connectivity": f"{network}_con_{age}_left"}, inplace=True)
        con_df_age_left = con_df_age_left[["subject", f"{network}_con_{age}_left"]]


        # handle outliers
        con_df_age_left=plot_and_remove_outliers(con_df_age_left, f"{network}_con_{age}_left",
                                                z_threshold=3, transform=False, connectivity = True)
        all_dfs.append(con_df_age_left)
        
        # Right hemisphere connectivity for the current age group
        con_df_age_right = con_df_region_gamma.loc[con_df_region_gamma.age == age]
        con_df_age_right = con_df_age_right[con_df_age_right[network]]
        con_df_age_right = con_df_age_right[con_df_age_right.hem_con == "right"]  # Filter right hemisphere
        con_df_age_right = con_df_age_right.groupby(["subject"]).mean(numeric_only=True).reset_index()
        con_df_age_right.rename(columns={"connectivity": f"{network}_con_{age}_right"}, inplace=True)
        con_df_age_right = con_df_age_right[["subject", f"{network}_con_{age}_right"]]

        # handle outliers
        con_df_age_right=plot_and_remove_outliers(con_df_age_right, f"{network}_con_{age}_right",
                                                   z_threshold=3, transform=False, connectivity = True)

        
        all_dfs.append(con_df_age_right)


In [ ]:
# Now merge all dataframes on the 'subject' column
final_df = all_dfs[0]
for df in all_dfs[1:]:
    if "empty" in df:
        continue
    final_df = final_df.merge(df, on="subject", how="outer")

In [ ]:
final_df

In [ ]:
final_df=final_df[[ 'subject',
                   'dmn_con_6', 'dmn_con_12',
                   'attn_con_6','attn_con_12',
                   'visual_con_6','visual_con_12', 
                   'auditory_con_6','auditory_con_12',
        'lang_comp_con_6','lang_comp_con_12',
        'speech_con_6_left','speech_con_12_left']]
final_df

## Behavioural assessment tools (Mullen Scales or Early Learning)

In [ ]:
behv_df.rename(columns={"id":"subject"}, inplace=True) # Rename the ID column to subject
behv_df['subject'] = behv_df['subject'].replace({'EC': ''}, regex=True) # Remove the EC from the subject ID


In [ ]:
# Check columns with Mullen and age equivalents due to floors effects (Akshoomoff, 2006; Munson et al., 2008).
behav_comlumns =[col for col in behv_df.columns if 'Mullen' and 'Age Equiv' in col] 
behv_df[behv_df[behav_comlumns]>300]=np.nan # Replace missingness codes 777, ,888, and 999 with NaN

In [ ]:
behav_comlumns.insert(0, "subject") # Insert subject as the first column
behv_df=behv_df[behav_comlumns] # Select only the columns with Mullen in the name
behv_df

In [ ]:
# Create a list to store the new column names
new_col_names = [] # 
for col in behav_comlumns:
    words = col.split()
    if len(words) < 2: # If the column name has only one word, keep it as it is (this is for subjects column)
        new_col_names.append(words[0])
    else:
        new_string = ' '.join([words[1]] + [words[-1]])
        new_string=new_string.lower().replace(" ", "_").replace("m","")
        new_col_names.append(new_string)

#new_col_names # Check the new column names

In [ ]:
print(len(behv_df.columns)) # Check the number of columns before renaming, sanity check
print(len(new_col_names))
behv_df.columns = new_col_names # Rename the columns

In [ ]:
# Calculate the nonverbal IQ scores for each age group
behv_df["nonverbal_iq_6"]=behv_df[["visual_6","fine_6"]].mean(axis=1)
behv_df["nonverbal_iq_12"]=behv_df[["visual_12","fine_12"]].mean(axis=1)
behv_df["nonverbal_iq_24"]=behv_df[["visual_24","fine_24"]].mean(axis=1)
behv_df["nonverbal_iq_36"]=behv_df[["visual_36","fine_36"]].mean(axis=1)

In [ ]:
#Calculate the expressive to receptive discrepancy score for each age group
behv_df["exp_rec_6"]=behv_df["expressive_6"]-behv_df["receptive_6"]
behv_df["exp_rec_12"]=behv_df["expressive_12"]-behv_df["receptive_12"]
behv_df["exp_rec_24"]=behv_df["expressive_24"]-behv_df["receptive_24"]
behv_df["exp_rec_36"]=behv_df["expressive_36"]-behv_df["receptive_36"]


In [ ]:
behv_df

#### Demographic information (Diagnosis, Risk, Gender)

In [ ]:
asd_df=demo_df.copy()
asd_df.subject= asd_df.subject.astype('int64')
asd_df

### Merge with clutering solutions

In [ ]:
clutering_solutions_df=clutering_solutions_df[["subject", "hc_class", "lpa_class",
                                               'receptive_6', 'expressive_6',
       'receptive_12', 'expressive_12', 'receptive_18', 'expressive_18',
       'receptive_24', 'expressive_24', 'receptive_36', 'expressive_36',
       'nonverbal_iq_6']]

### Create literature-driven dataset

In [ ]:
#Define the dataframes to be merged
dataframes = [frontal_power_6_sources, frontal_power_12_sources, frontal_power_18_sources,
              final_df,  lat_df_gamma_6, lat_df_gamma_12, lat_df_gamma_18,
              asd_df,clutering_solutions_df]

# Merge the dataframes based on the "subject" column
literature_driven_df = reduce(lambda left, right: pd.merge(left, right, on='subject', how='outer'), dataframes)


In [ ]:
literature_driven_df

In [ ]:
literature_driven_df.columns

In [ ]:
eeg_measures=['front_gamma_6', 'front_gamma_12', 'front_gamma_18',
       'dmn_con_6', 'dmn_con_12', 'attn_con_6', 'attn_con_12', 'visual_con_6',
       'visual_con_12', 'auditory_con_6', 'auditory_con_12', 'lang_comp_con_6',
       'lang_comp_con_12', 'speech_con_6_left', 'speech_con_12_left',
       'gamma_lat_6', 'gamma_lat_12', 'gamma_lat_18']

In [ ]:
# Drop all rows where there is missing EEG data
print("Shape before dropping EEG participants:", literature_driven_df.shape)
literature_driven_df.dropna(subset=eeg_measures, how="all", inplace=True)
print("Shape after dropping EEG participants:", literature_driven_df.shape)


### Final EEG dataset

In [ ]:
literature_driven_df.to_csv(f"../../datasets/sex_diff_main.csv", index=False)

In [ ]:
# Measures at 6 months
print ("Measures at 6 months", literature_driven_df.columns[literature_driven_df.columns.str.contains("6")])

In [ ]:
# Measures at 12 months
print ("Measures at 12 months", literature_driven_df.columns[literature_driven_df.columns.str.contains("12")])

In [ ]:
# Measures at 18 months
print ("Measures at 18 months", literature_driven_df.columns[literature_driven_df.columns.str.contains("18")])